# 🧪 SDV (Synthetic Data Vault) — คู่มือการใช้งานฉบับสมบูรณ์

Notebook นี้ครอบคลุม:
1. **Single Table** — สร้างข้อมูล synthetic จากตารางเดียว
2. **Multi Table** — สร้างข้อมูลจากหลายตารางที่มีความสัมพันธ์กัน
3. **Sequential / Time Series** — ข้อมูลที่มีลำดับเวลา
4. **Constraints** — กำหนด business rules
5. **Fine-tuning** — ปรับจูน model ในแบบต่างๆ
6. **Evaluation** — วัดคุณภาพข้อมูล synthetic
7. **Save / Load Model** — บันทึกและโหลด model

> **Install:** `pip install sdv`

In [ ]:
# ติดตั้ง library (ถ้ายังไม่ได้ติดตั้ง)
# !pip install sdv

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
print('✅ Import สำเร็จ')

---
## 📦 PART 1: Single Table Synthesis
### 1.1 โหลด Demo Dataset

In [ ]:
from sdv.datasets.demo import download_demo

real_data, metadata = download_demo(
    modality='single_table',
    dataset_name='fake_hotel_guests'
)

print('Shape:', real_data.shape)
real_data.head()

In [ ]:
# ดู metadata
metadata

### 1.2 สร้าง Metadata จาก DataFrame ของตัวเอง

In [ ]:
from sdv.metadata import SingleTableMetadata

# สร้าง DataFrame ตัวอย่าง
my_data = pd.DataFrame({
    'id':         range(1, 101),
    'age':        np.random.randint(18, 70, 100),
    'salary':     np.random.normal(50000, 15000, 100).round(2),
    'department': np.random.choice(['HR', 'IT', 'Finance', 'Sales'], 100),
    'join_date':  pd.date_range('2015-01-01', periods=100, freq='30D').astype(str),
    'is_manager': np.random.choice([True, False], 100)
})

# ให้ SDV ตรวจจับ types อัตโนมัติ
my_metadata = SingleTableMetadata()
my_metadata.detect_from_dataframe(my_data)

# กำหนด primary key
my_metadata.set_primary_key('id')

# แก้ไข type ถ้าจำเป็น
my_metadata.update_column(column_name='join_date', sdtype='datetime', datetime_format='%Y-%m-%d')

print(my_metadata)
my_metadata.visualize()

### 1.3 GaussianCopula Synthesizer (Model เร็ว, เหมาะกับข้อมูลทั่วไป)

In [ ]:
from sdv.single_table import GaussianCopulaSynthesizer

gc_synthesizer = GaussianCopulaSynthesizer(metadata)
gc_synthesizer.fit(real_data)

synthetic_gc = gc_synthesizer.sample(num_rows=200)
print('Synthetic data shape:', synthetic_gc.shape)
synthetic_gc.head()

### 1.4 CTGAN Synthesizer (Deep Learning, เหมาะกับข้อมูลซับซ้อน)

In [ ]:
from sdv.single_table import CTGANSynthesizer

ctgan_synthesizer = CTGANSynthesizer(
    metadata,
    epochs=100,          # จำนวน epochs (default=300, ลดลงเพื่อความเร็ว)
    batch_size=500,      # batch size
    verbose=True         # แสดง progress
)
ctgan_synthesizer.fit(real_data)

synthetic_ctgan = ctgan_synthesizer.sample(num_rows=200)
synthetic_ctgan.head()

### 1.5 TVAE Synthesizer (Variational Autoencoder)

In [ ]:
from sdv.single_table import TVAESynthesizer

tvae_synthesizer = TVAESynthesizer(
    metadata,
    epochs=100,
    batch_size=500
)
tvae_synthesizer.fit(real_data)

synthetic_tvae = tvae_synthesizer.sample(num_rows=200)
synthetic_tvae.head()

### 1.6 CopulaGAN Synthesizer (ผสม GaussianCopula + GAN)

In [ ]:
from sdv.single_table import CopulaGANSynthesizer

copula_gan = CopulaGANSynthesizer(
    metadata,
    epochs=100
)
copula_gan.fit(real_data)

synthetic_copulagan = copula_gan.sample(num_rows=200)
synthetic_copulagan.head()

---
## 🎛️ PART 2: Fine-Tuning — ปรับจูน Model

### 2.1 Fine-Tuning GaussianCopula — เลือก Distribution ของแต่ละ column

In [ ]:
from sdv.single_table import GaussianCopulaSynthesizer

# ดู numerical distributions ที่ใช้ได้
GaussianCopulaSynthesizer.get_available_distributions()

In [ ]:
# กำหนด distribution สำหรับ column เฉพาะ
gc_tuned = GaussianCopulaSynthesizer(
    metadata,
    # กำหนด default distribution
    default_distribution='beta',
    # กำหนด distribution แต่ละ column
    numerical_distributions={
        'checkin_date': 'uniform',
        'checkout_date': 'uniform',
        'room_rate': 'truncnorm',   # truncated normal
        'amenities_fee': 'beta'
    }
)

gc_tuned.fit(real_data)
synthetic_gc_tuned = gc_tuned.sample(num_rows=200)
synthetic_gc_tuned.head()

### 2.2 Fine-Tuning CTGAN — ปรับ Network Architecture

In [ ]:
from sdv.single_table import CTGANSynthesizer

ctgan_tuned = CTGANSynthesizer(
    metadata,
    # --- Training parameters ---
    epochs=300,                  # เพิ่ม epochs สำหรับข้อมูลซับซ้อน
    batch_size=500,              # batch size (ควรเป็น multiple ของ 10)
    
    # --- Network architecture ---
    generator_dim=(256, 256),    # ขนาด Generator layers
    discriminator_dim=(256, 256),# ขนาด Discriminator layers
    
    # --- Optimizer parameters ---
    generator_lr=2e-4,           # learning rate ของ Generator
    discriminator_lr=2e-4,       # learning rate ของ Discriminator
    generator_decay=1e-6,        # weight decay Generator
    discriminator_decay=1e-6,    # weight decay Discriminator
    
    # --- Training strategy ---
    discriminator_steps=1,       # จำนวน D steps ต่อ 1 G step
    
    # --- Regularization ---
    pac=10,                      # PacGAN packaging size
    
    verbose=True,
    cuda=False                   # ใช้ True ถ้ามี GPU
)

ctgan_tuned.fit(real_data)
synthetic_ctgan_tuned = ctgan_tuned.sample(num_rows=200)
synthetic_ctgan_tuned.head()

### 2.3 Fine-Tuning TVAE — ปรับ Encoder/Decoder

In [ ]:
from sdv.single_table import TVAESynthesizer

tvae_tuned = TVAESynthesizer(
    metadata,
    # --- Architecture ---
    compress_dims=(128, 128),    # Encoder dimensions
    decompress_dims=(128, 128),  # Decoder dimensions
    embedding_dim=128,           # ขนาด latent space
    
    # --- Training ---
    epochs=300,
    batch_size=500,
    lr=1e-3,                     # learning rate
    weight_decay=1e-5,
    loss_factor=2,               # weight ของ reconstruction loss
    
    cuda=False
)

tvae_tuned.fit(real_data)
synthetic_tvae_tuned = tvae_tuned.sample(num_rows=200)
synthetic_tvae_tuned.head()

### 2.4 Fine-Tuning: กำหนดวิธี Transform ของแต่ละ Column (Transformers)

In [ ]:
# ดู transformers ที่ใช้อยู่ใน model ปัจจุบัน
gc_synthesizer.get_transformers()

In [ ]:
from rdt.transformers import (
    FloatFormatter,
    LabelEncoder,
    UnixTimestampEncoder,
    BinaryEncoder
)

gc_transformer_tuned = GaussianCopulaSynthesizer(metadata)

# กำหนด transformer เฉพาะ column
gc_transformer_tuned.update_transformers({
    'room_type': LabelEncoder(add_noise=True),   # categorical → label + noise
    'amenities_fee': FloatFormatter(
        missing_value_replacement='mean',
        learn_rounding_scheme=True               # เรียนรู้การปัดเศษ
    )
})

gc_transformer_tuned.fit(real_data)
synthetic_gc_transformer = gc_transformer_tuned.sample(200)
synthetic_gc_transformer.head()

### 2.5 Fine-Tuning: Conditional Sampling — สุ่มข้อมูลแบบมีเงื่อนไข

In [ ]:
from sdv.sampling import Condition

# สร้างเงื่อนไข: ต้องการข้อมูล guest ที่จอง 'SUITE' เท่านั้น
suite_condition = Condition(
    num_rows=100,
    column_values={'room_type': 'SUITE'}
)

synthetic_conditional = gc_synthesizer.sample_from_conditions(
    conditions=[suite_condition]
)

print('Room types:', synthetic_conditional['room_type'].value_counts())
synthetic_conditional.head()

In [ ]:
# Multiple conditions
cond_suite_vip = Condition(
    num_rows=50,
    column_values={'room_type': 'SUITE', 'has_rewards': True}
)

cond_standard = Condition(
    num_rows=150,
    column_values={'room_type': 'STANDARD'}
)

synthetic_multi_cond = gc_synthesizer.sample_from_conditions(
    conditions=[cond_suite_vip, cond_standard]
)

print('Total rows:', len(synthetic_multi_cond))
print(synthetic_multi_cond['room_type'].value_counts())

### 2.6 Fine-Tuning: Anonymization — ปกป้องข้อมูลส่วนบุคคล

In [ ]:
from sdv.metadata import SingleTableMetadata

anon_metadata = SingleTableMetadata()
anon_metadata.detect_from_dataframe(real_data)
anon_metadata.set_primary_key('guest_email')

# กำหนด PII columns ให้ใช้ faker แทนค่าจริง
anon_metadata.update_column(
    column_name='guest_email',
    sdtype='email',
    pii=True
)
anon_metadata.update_column(
    column_name='billing_address',
    sdtype='address',
    pii=True
)
anon_metadata.update_column(
    column_name='credit_card_number',
    sdtype='credit_card_number',
    pii=True
)

gc_anon = GaussianCopulaSynthesizer(anon_metadata)
gc_anon.fit(real_data)
synthetic_anon = gc_anon.sample(100)
print('📧 Anonymized emails:')
print(synthetic_anon['guest_email'].head())

---
## ⛓️ PART 3: Constraints — กำหนด Business Rules

### 3.1 FixedCombinations — คู่ค่าที่ต้องออกมาด้วยกันเสมอ

In [ ]:
from sdv.single_table import GaussianCopulaSynthesizer

gc_constrained = GaussianCopulaSynthesizer(metadata)

# checkin_date ต้องมาก่อน checkout_date เสมอ
gc_constrained.add_constraints(constraints=[
    {
        'constraint_class': 'Inequality',
        'constraint_parameters': {
            'low_column_name': 'checkin_date',
            'high_column_name': 'checkout_date'
        }
    }
])

gc_constrained.fit(real_data)
synthetic_constrained = gc_constrained.sample(200)

# ตรวจสอบว่า constraint ถูกต้อง
valid = synthetic_constrained['checkin_date'] <= synthetic_constrained['checkout_date']
print(f'✅ Constraint valid: {valid.all()} ({valid.sum()}/{len(valid)} rows)')

### 3.2 Positive Values — ค่าต้องเป็นบวก

In [ ]:
gc_positive = GaussianCopulaSynthesizer(metadata)

gc_positive.add_constraints(constraints=[
    {
        'constraint_class': 'Positive',
        'constraint_parameters': {
            'column_name': 'room_rate',
            'strict': True  # > 0 (ไม่รวม 0)
        }
    },
    {
        'constraint_class': 'Positive',
        'constraint_parameters': {
            'column_name': 'amenities_fee',
            'strict': False  # >= 0 (รวม 0)
        }
    }
])

gc_positive.fit(real_data)
syn_positive = gc_positive.sample(200)
print('Min room_rate:', syn_positive['room_rate'].min())
print('Min amenities_fee:', syn_positive['amenities_fee'].min())

### 3.3 FixedCombinations — combination ของหลาย column ต้องเหมือนกับ real data

In [ ]:
gc_fixed = GaussianCopulaSynthesizer(metadata)

gc_fixed.add_constraints(constraints=[
    {
        'constraint_class': 'FixedCombinations',
        'constraint_parameters': {
            'column_names': ['room_type', 'has_rewards']  # combination ต้องออกมาแบบเดิม
        }
    }
])

gc_fixed.fit(real_data)
syn_fixed = gc_fixed.sample(200)

# ตรวจสอบ combinations
real_combos = set(zip(real_data['room_type'], real_data['has_rewards']))
syn_combos = set(zip(syn_fixed['room_type'], syn_fixed['has_rewards']))
print('Real combinations:', real_combos)
print('Syn combinations:', syn_combos)
print('✅ All valid:', syn_combos.issubset(real_combos))

---
## 🏛️ PART 4: Multi-Table Synthesis
### 4.1 โหลด Multi-Table Demo

In [ ]:
from sdv.datasets.demo import download_demo

multi_data, multi_metadata = download_demo(
    modality='multi_table',
    dataset_name='fake_hotels'
)

print('Tables:', list(multi_data.keys()))
for table, df in multi_data.items():
    print(f'  {table}: {df.shape}')

In [ ]:
# visualize relationships
multi_metadata.visualize()

### 4.2 HMASynthesizer (Hierarchical Modeling Algorithm)

In [ ]:
from sdv.multi_table import HMASynthesizer

hma_synthesizer = HMASynthesizer(multi_metadata)
hma_synthesizer.fit(multi_data)

# สร้างข้อมูล synthetic สำหรับทุก table
synthetic_multi = hma_synthesizer.sample(scale=1.0)  # scale=1.0 = ขนาดเท่ากับ real data

for table, df in synthetic_multi.items():
    print(f'{table}: {df.shape}')

### 4.3 Fine-Tuning HMA — กำหนด synthesizer แต่ละ table

In [ ]:
from sdv.multi_table import HMASynthesizer
from sdv.single_table import CTGANSynthesizer, GaussianCopulaSynthesizer

hma_tuned = HMASynthesizer(
    multi_metadata,
    # กำหนด synthesizer สำหรับแต่ละ table
    table_synthesizer_kwargs={
        'hotels': {
            'default_distribution': 'beta'
        },
        'guests': {
            'default_distribution': 'truncnorm'
        }
    }
)

hma_tuned.fit(multi_data)
synthetic_hma_tuned = hma_tuned.sample(scale=1.0)
print('✅ HMA Fine-tuned complete')

---
## 📅 PART 5: Sequential Data (Time Series)
### 5.1 โหลด Sequential Demo

In [ ]:
from sdv.datasets.demo import download_demo

seq_data, seq_metadata = download_demo(
    modality='sequential',
    dataset_name='nasdaq100_2019'
)

print('Shape:', seq_data.shape)
seq_data.head()

### 5.2 PARSynthesizer (Probabilistic AutoRegressive)

In [ ]:
from sdv.sequential import PARSynthesizer

par_synthesizer = PARSynthesizer(
    seq_metadata,
    epochs=128,
    verbose=True
)
par_synthesizer.fit(seq_data)

# sample ตาม context (เช่น ต้องการ 5 entities แต่ละ entity มี 252 timesteps)
synthetic_seq = par_synthesizer.sample(num_sequences=5)
print('Sequential data shape:', synthetic_seq.shape)
synthetic_seq.head()

### 5.3 Fine-Tuning PAR — ปรับ Network

In [ ]:
par_tuned = PARSynthesizer(
    seq_metadata,
    epochs=256,
    max_sequence_len=100,        # จำกัดความยาว sequence
    sample_size=5,               # จำนวน sequences ที่ sample ต่อ iteration
    context_columns=['Symbol'],  # ใช้ Symbol เป็น context ในการ generate
    verbose=True,
    cuda=False
)
par_tuned.fit(seq_data)

# Sample โดยกำหนด context
context_df = pd.DataFrame({'Symbol': ['AAPL', 'GOOGL', 'MSFT']})
synthetic_seq_ctx = par_tuned.sample(
    num_sequences=3,
    context=context_df
)
synthetic_seq_ctx.head()

---
## 📊 PART 6: Evaluation — วัดคุณภาพข้อมูล

### 6.1 Quality Report

In [ ]:
from sdv.evaluation.single_table import evaluate_quality

quality_report = evaluate_quality(
    real_data,
    synthetic_gc,
    metadata
)

print('\n📈 Overall Quality Score:', quality_report.get_score())

In [ ]:
# ดูรายละเอียดแต่ละ property
quality_report.get_details('Column Shapes')

In [ ]:
quality_report.get_details('Column Pair Trends')

### 6.2 Diagnostic Report — ตรวจหาปัญหา

In [ ]:
from sdv.evaluation.single_table import run_diagnostic

diagnostic = run_diagnostic(
    real_data,
    synthetic_gc,
    metadata
)

print('Diagnostic Score:', diagnostic.get_score())
diagnostic.get_details('Data Validity')

### 6.3 Visualization — เปรียบเทียบ Real vs Synthetic

In [ ]:
from sdv.evaluation.single_table import get_column_plot

# plot column เดียว
fig = get_column_plot(
    real_data=real_data,
    synthetic_data=synthetic_gc,
    column_name='amenities_fee',
    metadata=metadata
)
fig.show()

In [ ]:
from sdv.evaluation.single_table import get_column_pair_plot

# plot ความสัมพันธ์ระหว่าง 2 columns
fig = get_column_pair_plot(
    real_data=real_data,
    synthetic_data=synthetic_gc,
    column_names=['checkin_date', 'amenities_fee'],
    metadata=metadata
)
fig.show()

### 6.4 เปรียบเทียบหลาย Model

In [ ]:
from sdv.evaluation.single_table import evaluate_quality

models = {
    'GaussianCopula': synthetic_gc,
    'CTGAN': synthetic_ctgan,
    'TVAE': synthetic_tvae,
}

results = {}
for name, syn_data in models.items():
    report = evaluate_quality(real_data, syn_data, metadata, verbose=False)
    results[name] = report.get_score()
    print(f'{name}: {results[name]:.4f}')

best_model = max(results, key=results.get)
print(f'\n🏆 Best model: {best_model} (score={results[best_model]:.4f})')

---
## 💾 PART 7: Save & Load Model
### 7.1 บันทึก Model

In [ ]:
# บันทึก synthesizer ลงไฟล์
gc_synthesizer.save('gc_hotel_synthesizer.pkl')
ctgan_synthesizer.save('ctgan_hotel_synthesizer.pkl')
print('✅ Models saved!')

### 7.2 โหลด Model กลับมาใช้

In [ ]:
from sdv.single_table import GaussianCopulaSynthesizer

loaded_synthesizer = GaussianCopulaSynthesizer.load('gc_hotel_synthesizer.pkl')

# ใช้งานได้ทันทีโดยไม่ต้อง fit ใหม่
synthetic_from_loaded = loaded_synthesizer.sample(num_rows=100)
print('✅ Loaded & sampled:', synthetic_from_loaded.shape)
synthetic_from_loaded.head()

---
## 🚀 PART 8: SDV Lite — ใช้งานง่ายด้วย Single Line

สำหรับผู้เริ่มต้นที่ต้องการ auto-select model ที่ดีที่สุด

In [ ]:
from sdv.lite import SingleTablePreset

# FAST_ML preset — เร็วที่สุด เหมาะ prototype
fast_synthesizer = SingleTablePreset(
    metadata,
    name='FAST_ML'
)
fast_synthesizer.fit(real_data)
synthetic_fast = fast_synthesizer.sample(num_rows=200)
synthetic_fast.head()

---
## 📋 สรุป: เมื่อไหร่ควรใช้ Model ไหน?

| Model | เหมาะกับ | ข้อดี | ข้อเสีย |
|---|---|---|---|
| **GaussianCopula** | ข้อมูลทั่วไป, prototype | เร็ว, ตีความได้ | อาจไม่ดีกับ complex patterns |
| **CTGAN** | ข้อมูล imbalanced, complex | แม่นยำสูง | ช้า, ต้องการ GPU |
| **TVAE** | ข้อมูลที่มี latent structure | สมดุลระหว่างเร็วกับแม่น | ปรับ hyperparameter ยาก |
| **CopulaGAN** | ผสม Copula + GAN | ดีทั้ง distribution + correlation | ซับซ้อน |
| **HMA** | Multi-table | รักษา foreign key relationships | ช้าถ้า table ใหญ่ |
| **PAR** | Time series, sequential | เก่งเรื่อง temporal patterns | ต้องการข้อมูลเยอะ |
| **SingleTablePreset (FAST_ML)** | Quick prototype | ง่าย, ไม่ต้อง tune | ไม่ยืดหยุ่น |

### 🎯 Tips สำหรับการ Fine-tuning

1. **เริ่มด้วย GaussianCopula** เสมอ → baseline ที่เร็วและเสถียร
2. **เพิ่ม epochs** (CTGAN/TVAE) ถ้า quality ต่ำ แต่ระวัง overfitting
3. **ปรับ batch_size** ให้เหมาะกับขนาด dataset (500-2000)
4. **ใช้ Constraints** เสมอถ้ามี business rules
5. **ตรวจสอบด้วย evaluate_quality()** ทุกครั้งหลัง tune